# 🧩 Mini-Lab: Multi-Step Prompts

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what prompt chaining is and why breaking complex tasks into sequential steps improves quality
2. **Implement** a multi-step prompt chain where each step's output feeds into the next step's input
3. **Compare** a single monolithic prompt vs. a chained approach on the same task

## Target Concepts

| Concept | Description |
|---------|-------------|
| Prompt Chaining | Breaking a complex task into a sequence of simpler prompts where each step's output becomes input for the next step |

## Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env
MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(prompt, system="You are a helpful assistant."):
    """Send a single prompt and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

print("✅ Ready")

✅ Ready


## What Is Prompt Chaining?

**Prompt chaining** is a technique where you decompose a complex task into a sequence of simpler prompts. Each step produces an output that becomes the input for the next step.

```
User Input → [Step 1: Extract] → [Step 2: Analyze] → [Step 3: Format] → Final Output
```

**Why chain instead of doing it all in one prompt?**
- Each step is simpler, so the model makes fewer mistakes
- You can inspect and debug intermediate results
- You can use different instructions (or even models) per step
- Complex reasoning becomes more reliable when broken into stages

## Step 1 — The Monolithic Approach (Single Prompt)

Let's start by asking the model to do everything in one shot: take a product review, extract key points, analyze sentiment per point, and produce a structured summary.

In [2]:
review = """
I bought the ProMax Blender 3000 last month. The motor is incredibly powerful — it 
crushes ice in seconds and makes the smoothest smoothies I've ever had. The build 
quality feels premium with its stainless steel base. However, it's EXTREMELY loud, 
like jet-engine loud. My roommate complained from two rooms away. Also, the lid 
doesn't seal properly and I've had spinach smoothie splash onto my ceiling twice. 
Customer service was responsive when I reported the lid issue and they're sending 
a replacement. Overall, great blending power but the noise and lid design need work.
""".strip()

print(f"Review length: {len(review)} characters")

Review length: 582 characters


In [3]:
single_prompt = f"""Analyze this product review. Do ALL of the following:
1. Extract the key points mentioned
2. Classify each point as positive, negative, or neutral
3. Write a 2-sentence executive summary
4. Give an overall sentiment score from 1-10
5. List recommended improvements for the product team

Review:
{review}"""

single_result = chat(single_prompt)
md(single_result)

1. **Key Points Extracted:**
   - Powerful motor that crushes ice quickly.
   - Makes the smoothest smoothies.
   - Premium build quality with a stainless steel base.
   - Extremely loud operation.
   - Lid does not seal properly, causing spills.
   - Responsive customer service regarding the lid issue.
   - Replacement lid is being sent.

2. **Classification of Each Point:**
   - Powerful motor that crushes ice quickly: Positive
   - Makes the smoothest smoothies: Positive
   - Premium build quality with a stainless steel base: Positive
   - Extremely loud operation: Negative
   - Lid does not seal properly, causing spills: Negative
   - Responsive customer service regarding the lid issue: Positive
   - Replacement lid is being sent: Positive

3. **Executive Summary:**
   The ProMax Blender 3000 is praised for its powerful motor and premium build quality, delivering excellent smoothie results. However, it suffers from excessive noise and a poorly designed lid that causes spills, although customer service has been responsive in addressing the issue.

4. **Overall Sentiment Score:**
   7/10

5. **Recommended Improvements for the Product Team:**
   - Reduce the noise level of the blender to make it more user-friendly.
   - Redesign the lid to ensure a proper seal and prevent spills during use.
   - Consider including noise-dampening features or materials in future models.

That works, but the model had to juggle five sub-tasks at once. Let's see how chaining compares.

## Step 2 — The Chained Approach (Multi-Step Prompts)

We'll break the same analysis into **three focused steps**, passing each output forward.

```
Review → [Step 1: Extract key points] 
             → [Step 2: Sentiment per point + score] 
                  → [Step 3: Executive summary + recommendations]
```

### Chain Step 1: Extract Key Points

In [4]:
step1_prompt = f"""Extract the distinct key points from this product review.
List each point as a single bullet. Only extract — do not analyze or judge.

Review:
{review}"""

step1_output = chat(step1_prompt)

md("### Step 1 Output: Key Points\n" + step1_output)

### Step 1 Output: Key Points
- The ProMax Blender 3000 has a powerful motor that crushes ice quickly.
- It makes very smooth smoothies.
- The build quality is premium with a stainless steel base.
- The blender is extremely loud, comparable to a jet engine.
- The lid does not seal properly, causing spills.
- Customer service is responsive and is sending a replacement lid.
- Overall, the blender has great blending power but has issues with noise and lid design.

### Chain Step 2: Analyze Sentiment per Point

Now we feed Step 1's output into Step 2. The model only has to classify — it doesn't need to extract *and* classify at the same time.

In [5]:
step2_prompt = f"""For each key point below, classify the sentiment as Positive, Negative, or Neutral.
Then give an overall sentiment score from 1 (very negative) to 10 (very positive).

Key Points:
{step1_output}"""

step2_output = chat(step2_prompt)

md("### Step 2 Output: Sentiment Analysis\n" + step2_output)

### Step 2 Output: Sentiment Analysis
Let's classify the sentiment for each key point:

1. **The ProMax Blender 3000 has a powerful motor that crushes ice quickly.** - Positive
2. **It makes very smooth smoothies.** - Positive
3. **The build quality is premium with a stainless steel base.** - Positive
4. **The blender is extremely loud, comparable to a jet engine.** - Negative
5. **The lid does not seal properly, causing spills.** - Negative
6. **Customer service is responsive and is sending a replacement lid.** - Positive
7. **Overall, the blender has great blending power but has issues with noise and lid design.** - Neutral

Now, let's summarize the sentiments:
- Positive: 4 points
- Negative: 2 points
- Neutral: 1 point

To calculate the overall sentiment score:
- Positive points contribute positively, while negative points contribute negatively.
- We can assign a score of +1 for each positive sentiment and -1 for each negative sentiment.

Calculating the score:
- Positive points: 4 (4 * 1 = +4)
- Negative points: 2 (2 * -1 = -2)
- Neutral points do not affect the score.

Overall score = +4 - 2 = +2.

Now, we can map this score to a scale from 1 to 10:
- A score of +2 indicates a sentiment that is leaning towards negative but has some positive aspects.

Therefore, the overall sentiment score is **3**.

### Chain Step 3: Executive Summary & Recommendations

Finally, we combine all prior context to produce the summary. This step benefits from having clean, structured input from Steps 1 and 2.

In [6]:
step3_prompt = f"""Based on the key points and sentiment analysis below, produce:
1. A 2-sentence executive summary of the review
2. A bullet list of recommended improvements for the product team

Key Points:
{step1_output}

Sentiment Analysis:
{step2_output}"""

step3_output = chat(step3_prompt)

md("### Step 3 Output: Summary & Recommendations\n" + step3_output)

### Step 3 Output: Summary & Recommendations
### Executive Summary
The ProMax Blender 3000 is praised for its powerful motor and premium build quality, delivering exceptionally smooth smoothies. However, it suffers from significant noise levels and a poorly sealing lid, which can lead to spills, although customer service has been responsive in addressing the lid issue.

### Recommended Improvements for the Product Team
- **Reduce Noise Levels**: Investigate design modifications or materials that can help dampen the sound during operation.
- **Improve Lid Design**: Redesign the lid to ensure a proper seal to prevent spills during blending.
- **Enhance User Instructions**: Provide clearer guidance on lid placement and sealing to help users avoid issues.
- **Consider Sound Dampening Features**: Explore the addition of sound-dampening features or technology to make the blender quieter.
- **Quality Control Checks**: Implement stricter quality control measures to ensure that all lids fit properly before shipping.

## Step 3 — Building a Reusable Chain Function

In practice, you'd wrap prompt chains into a function so each step is clearly defined and the data flows automatically.

In [7]:
def analyze_review_chain(review_text, verbose=True):
    """Three-step prompt chain for review analysis."""
    
    # Step 1: Extract
    key_points = chat(
        f"Extract the distinct key points from this product review. "
        f"List each as a single bullet. Only extract — do not analyze.\n\n"
        f"Review:\n{review_text}"
    )
    if verbose:
        md("**Step 1 — Key Points:**\n" + key_points)
    
    # Step 2: Analyze
    sentiment = chat(
        f"Classify each key point as Positive, Negative, or Neutral. "
        f"Give an overall score from 1-10.\n\n"
        f"Key Points:\n{key_points}"
    )
    if verbose:
        md("---\n**Step 2 — Sentiment:**\n" + sentiment)
    
    # Step 3: Summarize
    summary = chat(
        f"Write a 2-sentence executive summary and list recommended improvements.\n\n"
        f"Key Points:\n{key_points}\n\n"
        f"Sentiment:\n{sentiment}"
    )
    if verbose:
        md("---\n**Step 3 — Summary & Recommendations:**\n" + summary)
    
    return {"key_points": key_points, "sentiment": sentiment, "summary": summary}

In [8]:
# Try the chain on a different review
new_review = """
The CloudWalk running shoes are super lightweight and the cushioning is amazing for 
long runs. My knee pain has actually decreased since switching to these. The breathable 
mesh keeps my feet cool. On the downside, the sizing runs small — I had to go up a 
full size. The color options are also quite limited, only black and grey. The price is 
fair for the quality you get.
""".strip()

result = analyze_review_chain(new_review)

**Step 1 — Key Points:**
- The CloudWalk running shoes are super lightweight.
- The cushioning is amazing for long runs.
- Knee pain has decreased since switching to these shoes.
- The breathable mesh keeps feet cool.
- Sizing runs small; needed to go up a full size.
- Limited color options; only available in black and grey.
- The price is fair for the quality.

---
**Step 2 — Sentiment:**
Here’s the classification of each key point along with an overall score:

1. **The CloudWalk running shoes are super lightweight.** - Positive
2. **The cushioning is amazing for long runs.** - Positive
3. **Knee pain has decreased since switching to these shoes.** - Positive
4. **The breathable mesh keeps feet cool.** - Positive
5. **Sizing runs small; needed to go up a full size.** - Negative
6. **Limited color options; only available in black and grey.** - Negative
7. **The price is fair for the quality.** - Positive

**Overall Classification:**
- Positive: 5 points
- Negative: 2 points
- Neutral: 0 points

**Overall Score: 8/10**

The overall score reflects a strong positive experience with some drawbacks regarding sizing and color options.

---
**Step 3 — Summary & Recommendations:**
**Executive Summary:**
The CloudWalk running shoes offer an exceptional lightweight design and outstanding cushioning, significantly improving comfort during long runs and reducing knee pain. While the shoes provide excellent quality at a fair price, they face challenges with small sizing and limited color options.

**Recommended Improvements:**
1. Expand sizing options to accommodate a wider range of foot sizes.
2. Introduce additional color choices to appeal to a broader audience.
3. Consider providing a sizing guide or recommendations to help customers select the correct size.
4. Explore options for custom designs or limited edition colors to enhance market appeal.

## When to Use Prompt Chaining

| Scenario | Single Prompt | Chaining |
|----------|:---:|:---:|
| Simple, one-step task | ✅ | Overkill |
| Task with 2+ distinct subtasks | ⚠️ | ✅ |
| Need to inspect intermediate results | ❌ | ✅ |
| Different instructions per step | ❌ | ✅ |
| Minimizing API calls matters | ✅ | ⚠️ More calls |

**Trade-off:** Chaining uses more API calls (and thus more cost/latency), but each call is simpler and more reliable. For complex tasks, the quality improvement is usually worth it.

## 🎯 Summary

### Key Takeaways

1. **Prompt chaining** — breaks a complex task into a sequence of simpler prompts where each step's output feeds into the next step's input
2. **Decomposition improves quality** — focused single-task prompts produce more reliable results than one monolithic prompt trying to do everything
3. **Debuggability** — chaining lets you inspect intermediate outputs so you can identify and fix exactly where things go wrong
4. **Reusable patterns** — wrapping chains into functions creates reliable, repeatable multi-step workflows